In [ ]:
# --- Importaciones ---------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA

import sklearn
print(f'numpy:   {np.__version__}')
print(f'pandas:  {pd.__version__}')
print(f'sklearn: {sklearn.__version__}')

In [ ]:
# --- Carga del dataset Mall Customers -------------------------------
url = ('https://raw.githubusercontent.com/dsrscientist/dataset1/master/Mall_Customers.csv')
try:
    df = pd.read_csv(url)
except Exception as e:
    print(f'Error al cargar ({e}). Creando dataset sintético...')
    rng = np.random.default_rng(42)
    df = pd.DataFrame({
        'CustomerID': range(1, 201),
        'Genre': rng.choice(['Male', 'Female'], 200),
        'Age': rng.integers(18, 70, 200),
        'Annual Income (k$)': rng.integers(15, 140, 200),
        'Spending Score (1-100)': rng.integers(1, 100, 200),
    })

print(df.shape)          # (200, 5)
print(df.head())
print(df.isnull().sum()) # verificar nulos

In [ ]:
# --- Seleccionar features para clustering ---------------------------
# Usamos Income y Spending Score para visualización en 2D
X_2d = df[['Annual Income (k$)', 'Spending Score (1-100)']].values

# --- Escalado: K-Means es sensible a la escala ----------------------
scaler = StandardScaler()
X_2d_sc = scaler.fit_transform(X_2d)

print(f'Media antes de escalar:  {X_2d.mean(axis=0).round(2)}')
print(f'Media después de escalar: {X_2d_sc.mean(axis=0).round(4)}')

In [ ]:
# --- K-Means con k=5 ------------------------------------------------
semilla = 42
km5 = KMeans(
    n_clusters=5,
    init='k-means++',   # inicialización inteligente (defecto)
    n_init=10,          # 10 inicializaciones aleatorias, toma la mejor
    random_state=semilla,
)
km5.fit(X_2d_sc)

etiquetas = km5.labels_          # cluster asignado a cada punto
centroides = km5.cluster_centers_  # coordenadas de los 5 centroides

print(f'Inercia: {km5.inertia_:.2f}')
print(f'Iteraciones hasta convergencia: {km5.n_iter_}')

# Distribución de puntos por cluster
for c in range(5):
    n = (etiquetas == c).sum()
    print(f'Cluster {c}: {n} clientes')

In [ ]:
# --- Curva de inercia (método del codo) -----------------------------
inercias = []
rango_k = range(1, 12)

for k in rango_k:
    km = KMeans(n_clusters=k, n_init=10, random_state=semilla)
    km.fit(X_2d_sc)
    inercias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(rango_k, inercias, 'o-', color='steelblue')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inercia')
plt.title('Método del codo — Mall Customers')
plt.xticks(rango_k)
plt.tight_layout()
plt.show()


In [ ]:
# --- Silhouette score para k=2..9 -----------------------------------
silhouettes = []

for k in range(2, 10):
    km = KMeans(n_clusters=k, n_init=10, random_state=semilla)
    etiq = km.fit_predict(X_2d_sc)
    try:
        sil = silhouette_score(X_2d_sc, etiq)
    except ValueError as e:
        print(f'k={k} silhouette no válido: {e}')
        sil = np.nan
    silhouettes.append(sil)
    print(f'k={k} | Silhouette: {sil:.4f}')

k_optimo = range(2, 10)[silhouettes.index(max(silhouettes))]
print(f'k óptimo según silhouette: {k_optimo}')

In [ ]:
# --- Visualizar el clustering óptimo --------------------------------
km_opt = KMeans(n_clusters=5, n_init=10, random_state=semilla)
etiq_opt = km_opt.fit_predict(X_2d_sc)
centroides_opt = km_opt.cluster_centers_

colores = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#9B5DE5']
nombres_cluster = [
    'Alto ingreso / Gasto alto',
    'Ingreso medio / Gasto medio',
    'Bajo ingreso / Gasto bajo',
    'Alto ingreso / Gasto bajo',
    'Bajo ingreso / Gasto alto',
]

fig, ax = plt.subplots(figsize=(8, 6))
for c in range(5):
    mask = etiq_opt == c
    ax.scatter(
        X_2d_sc[mask, 0], X_2d_sc[mask, 1],
        c=colores[c], label=nombres_cluster[c],
        s=60, alpha=0.8,
    )
ax.scatter(
    centroides_opt[:, 0], centroides_opt[:, 1],
    c='black', marker='X', s=200, zorder=5, label='Centroides',
)
ax.set_xlabel('Annual Income (estandarizado)')
ax.set_ylabel('Spending Score (estandarizado)')
ax.set_title('Segmentación de clientes — K-Means (k=5)')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- Dataset sintético con forma no convexa -------------------------
from sklearn.datasets import make_moons

rng = np.random.default_rng(semilla)
X_moons, y_moons = make_moons(n_samples=300, noise=0.08,
                            random_state=semilla)

# K-Means falla con esta forma (asume clusters esféricos)
km_moons = KMeans(n_clusters=2, n_init=10, random_state=semilla)
etiq_km = km_moons.fit_predict(X_moons)

# DBSCAN encuentra la forma correcta
db = DBSCAN(eps=0.2, min_samples=5)
etiq_db = db.fit_predict(X_moons)

n_clusters_db = len(set(etiq_db)) - (1 if -1 in etiq_db else 0)
n_ruido = (etiq_db == -1).sum()
print(f'Clusters encontrados: {n_clusters_db}')
print(f'Puntos de ruido:      {n_ruido}')
# Resultados típicos:
# Clusters encontrados: 2
# Puntos de ruido:      0
# --- Comparativa visual K-Means vs DBSCAN ---------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(X_moons[:, 0], X_moons[:, 1],
            c=etiq_km, cmap='Set1', s=30)
ax1.set_title('K-Means (k=2) — falla en formas no convexas')

# etiq_db: -1 = ruido (gris), 0/1/... = clusters (colores)
ax2.scatter(X_moons[:, 0], X_moons[:, 1],
            c=etiq_db, cmap='Set1', s=30)
ax2.set_title('DBSCAN — detecta la forma correcta')

plt.tight_layout()
plt.show()

In [ ]:
# --- Sensibilidad de DBSCAN a sus hiperparámetros ------------------
configuraciones = [
    (0.1, 5),   # eps muy pequeño -> muchos outliers
    (0.2, 5),   # configuración óptima
    (0.5, 5),   # eps grande -> un solo cluster
    (0.2, 15),  # min_samples alto -> más outliers
]

for eps_val, min_s in configuraciones:
    db_test = DBSCAN(eps=eps_val, min_samples=min_s)
    etiq_test = db_test.fit_predict(X_moons)
    n_cl = len(set(etiq_test)) - (1 if -1 in etiq_test else 0)
    n_ru = (etiq_test == -1).sum()
    print(
        f'eps={eps_val} min_samples={min_s:2d}',
        f'-> clusters={n_cl} ruido={n_ru}',
    )

In [ ]:
# --- Usar todas las features numéricas ------------------------------
X_full = df[['Age', 'Annual Income (k$)',
            'Spending Score (1-100)']].values
X_full_sc = StandardScaler().fit_transform(X_full)

km_full = KMeans(n_clusters=5, n_init=10, random_state=semilla)
etiq_full = km_full.fit_predict(X_full_sc)

sil_full = silhouette_score(X_full_sc, etiq_full)
print(f'Silhouette (3 features): {sil_full:.4f}')

# --- PCA para reducir a 2D y visualizar ----------------------------
pca = PCA(n_components=2, random_state=semilla)
X_pca = pca.fit_transform(X_full_sc)
var_explicada = pca.explained_variance_ratio_.sum()
print(f'Varianza explicada por 2 componentes: {var_explicada:.2%}')

plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=etiq_full, cmap='Set2', s=60, alpha=0.8
)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('Clusters en espacio PCA — Mall Customers (3 features)')
plt.tight_layout()
plt.show()

In [ ]:
# --- DBSCAN sobre Mall Customers 2D ---------------------------------
db_mall = DBSCAN(eps=0.4, min_samples=5)
etiq_db_mall = db_mall.fit_predict(X_2d_sc)

n_clusters_db_mall = len(set(etiq_db_mall)) - (1 if -1 in etiq_db_mall else 0)
n_ruido_mall = (etiq_db_mall == -1).sum()

print(f'DBSCAN: {n_clusters_db_mall} clusters, {n_ruido_mall} outliers')

# Silhouette solo si hay más de 1 cluster y hay puntos no-ruido
if n_clusters_db_mall > 1:
    mascara = etiq_db_mall != -1
    sil_dbscan = silhouette_score(
        X_2d_sc[mascara],
        etiq_db_mall[mascara],
    )
    print(f'Silhouette DBSCAN (sin outliers): {sil_dbscan:.4f}')

sil_kmeans = silhouette_score(X_2d_sc, km5.labels_)
print(f'Silhouette K-Means (k=5):        {sil_kmeans:.4f}')